In [1]:
import pennylane as qml
from pennylane import numpy as np

## Different forms of measurement 

- `qml.sample`: returns a sample from the *n* possible states 
- `qml.counts`: returns the number of counts for each sample
- `qml.probs`: returns the probability of each computational basis state
- `qml.expval`: returns the expectation value of the supplied observable

#### Using `qml.sample`

Measuring in the computational basis

In [2]:
dev = qml.device("default.qubit",wires=1, shots=10)

@qml.qnode(dev)
def circuit():
    qml.H(wires=0)
    return qml.sample()

print(circuit())

[[1]
 [1]
 [1]
 [1]
 [1]
 [0]
 [1]
 [0]
 [1]
 [0]]


/Users/vegf/Desktop/vscode/pennylane/.venv/lib/python3.12/site-packages/pennylane/devices/device_api.py:193: PennyLaneDeprecationWarning: Setting shots on device is deprecated. Please use the `set_shots` transform on the respective QNode instead.
  warnings.warn(


Measuring in the Hadamard basis, i.e. $\{\ket{+}, \ket{-}\}$

In [3]:
dev2 = qml.device("default.qubit",wires=2, shots=10)

@qml.qnode(dev2)
def pauliX_circuit():
    qml.H(wires=1)
    return qml.sample(qml.X(1))

print(pauliX_circuit())

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


#### Using `qml.counts`

Measuring in the computational basis

In [4]:
@qml.qnode(dev)
def circuit():
    qml.H(wires=0)
    return qml.counts()

print(circuit())

{np.str_('0'): np.int64(4), np.str_('1'): np.int64(6)}


Measuring in the Hadamard basis

In [5]:
@qml.qnode(dev2)
def pauliX_circuit():
    qml.H(wires=1)
    return qml.counts(qml.X(1))

print(pauliX_circuit())

{np.float64(1.0): np.int64(10)}


#### Using `qml.probs` 
(used without shots)

Measuring in the computational basis

In [6]:
dev_prob = qml.device("default.qubit",wires=1)

@qml.qnode(dev_prob)
def circuit():
    qml.H(wires=0)
    return qml.probs()

print(circuit())

[0.5 0.5]


Given a circuit with more than one qubit, we can check the probability of some qubits

In [7]:
dev3_prob = qml.device("default.qubit",wires=3)

@qml.qnode(dev3_prob)
def circuit():
    qml.H(wires=0)
    qml.X(wires=1)
    qml.X(wires=2)
    #return qml.probs(wires=0) #[0.5, 0.5]
    #return qml.probs(wires=1) #[0, 1]
    #return qml.probs(wires=2) #[1, 0]
    #return qml.probs(wires=[1,2])
    return qml.probs()

print(circuit())

[0.  0.  0.  0.5 0.  0.  0.  0.5]


Measuring in the Hadamard basis

In [8]:
@qml.qnode(dev_prob)
def circuit():
    qml.H(wires=0)
    return qml.probs(op=qml.X(0))

print(circuit())

[1. 0.]


In [9]:
dev2_prob = qml.device("default.qubit",wires=2)

@qml.qnode(dev2_prob)
def circuit():
    qml.X(wires=1)

    #qml.X(0)@qml.X(1) represents X \otimes X
    #we need to use op in qml.probs(...) because the first argument of qml.probs(...) is wires;
    #if we do not specify op, Pennylane will try to read the operator as a wire, and give an error
    return qml.probs(op = qml.X(0)@qml.X(1))
    #return qml.probs(op = qml.X(1))

print(circuit())


[0.25 0.25 0.25 0.25]


#### Using `qml.expval`

An **observable** is an experiment

When we calculate the expectation value over a state $\ket{\psi}$, we are calculating the average value of this variable over many measurements

For instance, the expectation value of the observable $\hat{Z}$ under the state $\ket{0}$ is given as follows:
    $$\langle \hat{Z} \rangle = \bra{0} \hat{Z} \ket{0} = 1$$

In [10]:
@qml.qnode(dev2_prob)
def circuit():
    qml.X(1)
    #return qml.expval(qml.Z(0))
    return qml.expval(qml.Z(0)@qml.I(1))

print(circuit())


1.0


The expectation value 

In [11]:
@qml.qnode(dev2_prob)
def circuit():
    qml.RY(np.pi/4, 0)
    qml.RX(np.pi/3, 1)
    qml.CNOT([0,1])

    return qml.expval(1/3 * qml.Z(0)@qml.Z(1))

print(circuit())

0.1666666666666667


## Multiple measurements

In [12]:
@qml.qnode(dev_prob)
def circuit():
    qml.H(0)

    return qml.expval(qml.Z(0)), qml.probs()

print(circuit())

(np.float64(0.0), array([0.5, 0.5]))
